# Weee! Review Fetcher (Colab Edition) 🛍️

Easily fetch product reviews from Weee! and export them to Excel with high-res images.

**Instructions:**
1. Run the **Setup** cell to install dependencies.
2. Configure your search in the **Configuration** cell.
3. Run the **Fetch & Download** cell to get your data.

## 1. Setup
Run this cell to install the required Python libraries.

In [ ]:
!pip install requests openpyxl pillow tqdm

## 2. Configuration & Execution
Fill in the product URL or ID, select your language, and set the maximum number of reviews.

In [ ]:
# @title Fetch Reviews
product_url_or_id = "https://www.sayweee.com/zh/products/HADAY-Delicious-Light-Soy-Sauce/113281" # @param {type:"string"}
language = "cn" # @param ["cn", "en"]
max_reviews_to_fetch = 50 # @param {type:"integer"}

import requests
import json
import time
import io
import os
import re
from openpyxl import Workbook
from openpyxl.drawing.image import Image as OpenpyxlImage
from openpyxl.utils import get_column_letter
from PIL import Image as PILImage
from tqdm.notebook import tqdm
from google.colab import files

def get_headers(product_id, lang="zh"):
    user_agent = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    if lang == "zh":
        accept_language = "zh-CN,zh;q=0.9,en;q=0.8"
        referer = f"https://www.sayweee.com/zh/product/view/{product_id}"
    else:
        accept_language = "en-US,en;q=0.9,zh;q=0.8"
        referer = f"https://www.sayweee.com/en/product/view/{product_id}"
    return {
        "User-Agent": user_agent,
        "Accept-Language": accept_language,
        "lang": lang,
        "Referer": referer,
    }

def fetch_all_reviews(product_id, max_reviews=None, lang='zh'):
    base_url = "https://api.sayweee.net/ec/social/review"
    all_reviews = []
    page = 1
    limit = 20
    
    pbar = tqdm(desc="Fetching reviews", unit="review")
    while True:
        params = {"product_id": product_id, "sort": "relevance", "page": page, "limit": limit}
        try:
            response = requests.get(base_url, params=params, headers=get_headers(product_id, lang))
            data = response.json()
            object_data = data.get('object') or {}
            reviews = object_data.get('list', [])
            
            if page == 1:
                total_from_api = object_data.get('total')
                if total_from_api:
                    pbar.total = min(total_from_api, max_reviews) if max_reviews else total_from_api
                    pbar.refresh()
            
            if not reviews: break
            
            added_reviews = min(len(reviews), max_reviews - len(all_reviews) if max_reviews else len(reviews))
            all_reviews.extend(reviews[:added_reviews])
            pbar.update(added_reviews)
            
            if max_reviews is not None and len(all_reviews) >= max_reviews: break
            page += 1
            time.sleep(0.5)
        except Exception as e:
            print(f"Error at page {page}: {e}")
            break
    pbar.close()
    return all_reviews

# 1. Parse Input
product_name = ""
if product_url_or_id.startswith('http'):
    parts = product_url_or_id.split('?')[0].strip('/').split('/')
    product_id = parts[-1]
    product_name = parts[-2] if len(parts) >= 2 else ""
else:
    product_id = product_url_or_id

target_lang = "zh" if language == "cn" else "en"
if product_name:
    product_name = re.sub(r'[\\/:*?"<>| ]', '-', product_name)

# 2. Fetch Data
reviews_data = fetch_all_reviews(product_id, max_reviews=max_reviews_to_fetch, lang=target_lang)

# 3. Save Folder
current_time = time.strftime("%Y%m%d_%H%M%S")
dir_name = f"{product_name}_{product_id}_{current_time}_{len(reviews_data)}" if product_name else f"{product_id}_{current_time}_{len(reviews_data)}"
os.makedirs(dir_name, exist_ok=True)

json_path = os.path.join(dir_name, f"{dir_name}.json")
excel_path = os.path.join(dir_name, f"{dir_name}.xlsx")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(reviews_data, f, ensure_ascii=False, indent=4)

if reviews_data:
    print("\nCreating Excel with images...")
    wb = Workbook()
    ws = wb.active
    ws.title = "Reviews"
    
    fieldnames = []
    for review in reviews_data:
        for key in review:
            if key not in fieldnames: fieldnames.append(key)

    max_pics = 0
    for review in reviews_data:
        pics = review.get('pictures')
        if isinstance(pics, list) and len(pics) > max_pics: max_pics = len(pics)

    if 'pictures' in fieldnames: fieldnames.remove('pictures')
    headers = fieldnames + [f"picture_{i+1}" for i in range(max_pics)]
    ws.append(headers)

    for i in range(max_pics):
        ws.column_dimensions[get_column_letter(len(fieldnames) + 1 + i)].width = 36

    for row_idx, review in enumerate(tqdm(reviews_data, desc="Processing Excel rows"), start=2):
        ws.row_dimensions[row_idx].height = 190
        for col_idx, key in enumerate(fieldnames, start=1):
            val = review.get(key)
            ws.cell(row=row_idx, column=col_idx).value = str(val) if isinstance(val, (dict, list)) else val
        
        pictures = review.get('pictures')
        if isinstance(pictures, list):
            for pic_idx, img_url in enumerate(pictures):
                col_idx = len(fieldnames) + 1 + pic_idx
                if img_url.startswith('//'): img_url = 'https:' + img_url
                try:
                    img_res = requests.get(img_url, timeout=5)
                    if img_res.status_code == 200:
                        img_stream = io.BytesIO(img_res.content)
                        pil_img = PILImage.open(img_stream)
                        w, h = pil_img.size
                        ratio = min(500/w, 500/h) if w > 0 and h > 0 else 1
                        img_byte_arr = io.BytesIO()
                        pil_img.save(img_byte_arr, format='PNG')
                        img_byte_arr.seek(0)
                        xl_img = OpenpyxlImage(img_byte_arr)
                        xl_img.width = int(w * ratio)
                        xl_img.height = int(h * ratio)
                        xl_img.anchor = ws.cell(row=row_idx, column=col_idx).coordinate
                        ws.add_image(xl_img)
                except: pass
    
    wb.save(excel_path)
    print(f"\nSuccess! Files saved in: {dir_name}")
    
    # 4. ZIP and Download
    import shutil
    shutil.make_archive(dir_name, 'zip', dir_name)
    print(f"\nZipping folder and starting download...")
    files.download(f"{dir_name}.zip")
else:
    print("No reviews found.")